# Secure Unity Catalog objects

## Demo Scenario: RetailNow – Omnichannel Retail Platform

**Industry Context: Retail**

RetailNow is an omnichannel retailer operating 80+ stores and an e-commerce platform across four US regions (North, South, East, West). Their Azure Databricks data platform stores sensitive customer and transactional data subject to privacy regulations.

**The security challenge**: Different stakeholders need different levels of access:
- **Retail analysts**: Company-wide query access, but must not see customer PII (phone, email)
- **Regional store managers**: Can only view transactions from their own region
- **Finance team**: Full access including PII for compliance and audit reporting
- **Automated pipelines**: Authenticate without hardcoded credentials

As Azure Databricks Data Engineers, we will demonstrate how Unity Catalog enables fine-grained, governed access to retail data without duplicating tables or managing dozens of views.

**Sample data in `trainer_demo.demo_04`:**
| Table | Description |
|---|---|
| `customers` | Customer profiles with PII (email, phone) and loyalty segment |
| `products` | Product catalog with pricing and cost margins |
| `stores` | Physical store locations by region |
| `transactions` | Sales across all stores linking customers and products |


### Setup: Load Sample Retail Data

Run the following cell to create sample tables in `trainer_demo.demo_04`:
- **customers**: 500 customer records including email, phone (PII fields), region, and loyalty segment
- **products**: 20-item product catalog across Electronics, Clothing, Sports, Home, Food, and Beauty
- **stores**: 16 physical locations grouped into North / South / East / West regions
- **transactions**: 5,000 sales records linking customers, products, and stores


In [ ]:
# Run the setup script to create sample retail data
from scripts.setup import setup
setup(spark)


In [ ]:
%sql
-- Set working context to the retail demo schema
USE CATALOG trainer_demo;
USE SCHEMA demo_04;

-- Preview customer data – note the PII columns (email, phone)
SELECT * FROM customers LIMIT 5;


## Understand query lifecycle

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/secure-unity-catalog-objects.understand-query-lifecycle.png)

### 🎯 DEMO: Observe the Query Lifecycle

When a principal queries a Unity Catalog table, the request flows through several security checkpoints before any data is returned:

1. **Query submitted** to the cluster or SQL warehouse
2. **Unity Catalog validates** the principal's privileges (USE CATALOG → USE SCHEMA → SELECT)
3. **Cloud credentials are assumed** — Unity Catalog generates a short-lived scoped token
4. **Data is read** from ADLS Gen2 using the token
5. **Row/column filters are applied** on the compute resource
6. **Result is returned** to the caller — the audit event is already written

Let's verify who is running this notebook and confirm the current data context.


In [ ]:
%sql
-- Identify the current principal and context
SELECT
  current_user()    AS current_user,
  current_catalog() AS catalog,
  current_schema()  AS schema;


In [ ]:
%sql
    
-- Unity Catalog records every data access in the system audit log.
-- The query below shows recent activity against our retail tables.
-- Note: requires READ privilege on system.access.audit (typically metastore admin).
SELECT
  event_time,
  user_identity.email AS user_name,
  action_name,
  request_params.commandText
FROM system.access.audit
WHERE action_name IN ('commandSubmit', 'runCommand')
  AND event_date >= CURRENT_DATE - INTERVAL 10 DAY
ORDER BY event_time DESC
LIMIT 10;

## Implement access control strategies

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/secure-unity-catalog-objects.implement-access-control-strategies.png)

![diagram](https://learn.microsoft.com/en-us/training/wwl-databricks/secure-unity-catalog-objects/media/understand-access-control-lists.png)

### 🎯 DEMO: Grant and Revoke Privileges

RetailNow has three user groups with different data access needs:

| Group | Description | Required Privileges |
|---|---|---|
| `retail_analysts` | Analytics team – insights across the business | USE CATALOG, USE SCHEMA, SELECT on all tables (PII masked) |
| `store_managers` | Regional managers – operational reporting | USE CATALOG, USE SCHEMA, SELECT (row-filtered per region) |
| `finance_team` | Finance & compliance – full audit access | USE CATALOG, USE SCHEMA, SELECT all tables including PII |

**Privilege inheritance reminder:**
- Granting at **table level** = explicit, most restrictive
- Granting **SELECT on SCHEMA** = all current and future tables inherit
- Granting **ALL PRIVILEGES on CATALOG** = broadest access

> **Trainer Note**: Groups are created in the Databricks Account Console or via SCIM/Entra ID sync. The GRANT statements below assume the groups already exist.


In [ ]:
%sql

--CREATE GROUP retail_analysts;
--CREATE GROUP finance_team;
--CREATE GROUP store_managers;

In [ ]:
%sql
USE CATALOG trainer_demo;
USE SCHEMA demo_04;

-- Grant traverse permissions at catalog and schema level
GRANT USE CATALOG ON CATALOG trainer_demo TO `retail_analysts`;
GRANT USE SCHEMA  ON SCHEMA  trainer_demo.demo_04 TO `retail_analysts`;

-- Grant table-level SELECT (explicit model – most restrictive)
GRANT SELECT ON TABLE trainer_demo.demo_04.customers     TO `retail_analysts`;
GRANT SELECT ON TABLE trainer_demo.demo_04.transactions  TO `retail_analysts`;
GRANT SELECT ON TABLE trainer_demo.demo_04.products      TO `retail_analysts`;
GRANT SELECT ON TABLE trainer_demo.demo_04.stores        TO `retail_analysts`;


In [ ]:
%sql
-- Verify grants on the customers table
SHOW GRANTS ON TABLE trainer_demo.demo_04.customers;


In [ ]:
%sql
-- Alternative: grant SELECT at schema level – inherited by ALL current and future tables.
-- Lower admin overhead, but broader access.
GRANT USE CATALOG  ON CATALOG trainer_demo        TO `finance_team`;
GRANT USE SCHEMA   ON SCHEMA  trainer_demo.demo_04 TO `finance_team`;
GRANT SELECT       ON SCHEMA  trainer_demo.demo_04 TO `finance_team`;

-- Revoke example: remove table-level access from retail_analysts for a sensitive table
-- REVOKE SELECT ON TABLE trainer_demo.demo_04.customers FROM `retail_analysts`;

-- Check all grants on the schema
SHOW GRANTS ON SCHEMA trainer_demo.demo_04;


## Understand fine-grained access control

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/secure-unity-catalog-objects.understand-fine-grained-access-control.png)

### 🎯 DEMO: Choosing Between Fine-Grained Access Control Approaches

Granting SELECT on the entire `customers` table still exposes PII to every analyst. Unity Catalog offers two approaches to restrict which rows and columns a principal can see:

**Approach A – Row and Column Security** *(preferred)*
- Attach masking functions directly to sensitive columns
- Attach a filter function directly to the table
- All queries automatically enforce the rules – no extra objects needed
- Fewer items to goverd, audit, and maintain

**Approach B – Dynamic Views**
- Create a view with `CASE WHEN is_account_group_member(...)` logic
- Users query the view; the underlying table is never directly exposed
- Useful when the view needs to join multiple tables, or strict abstraction is required

For RetailNow, we will demonstrate both: **column masking + row filtering** on the base tables, and a **dynamic view** as an alternative.


## Implement row filtering and column masking

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/secure-unity-catalog-objects.implement-row-filtering-column-masking.png)

### 🎯 DEMO: Column Masking – Protect Customer PII

`retail_analysts` need to query `customers` for business insights, but they must not see full phone numbers or email addresses. We create masking functions that reveal sensitive values only to members of `finance_team`.


In [ ]:
%sql
USE CATALOG trainer_demo;
USE SCHEMA demo_04;

-- Masking function for phone: shows last 4 digits to non-finance users
CREATE OR REPLACE FUNCTION mask_phone(phone STRING)
  RETURNS STRING
  COMMENT 'Masks phone number – finance_team sees full value; others see XXX-XXX-NNNN'
  RETURN CASE
    WHEN is_account_group_member('finance_team') THEN phone
    ELSE 'XXX-XXX-' || RIGHT(phone, 4)
  END;

-- Masking function for email: shows first 2 chars + domain to non-finance users
CREATE OR REPLACE FUNCTION mask_email(email STRING)
  RETURNS STRING
  COMMENT 'Masks email address – finance_team sees full value; others see partial'
  RETURN CASE
    WHEN is_account_group_member('finance_team') THEN email
    ELSE CONCAT(LEFT(email, 2), '***@', SPLIT(email, '@')[1])
  END;


In [ ]:
%sql
-- Attach the masking functions to the sensitive columns in customers
ALTER TABLE trainer_demo.demo_04.customers
  ALTER COLUMN phone
  SET MASK trainer_demo.demo_04.mask_phone;

ALTER TABLE trainer_demo.demo_04.customers
  ALTER COLUMN email
  SET MASK trainer_demo.demo_04.mask_email;


In [ ]:
%sql
-- Query customers – phone and email are now automatically masked for non-finance users.
-- Running as admin you see real values; retail_analysts would see redacted output.
SELECT customer_id, first_name, last_name, email, phone, customer_segment, region
FROM trainer_demo.demo_04.customers
LIMIT 10;


### 🎯 DEMO: Row Filtering – Regional Transaction Access

Store managers should only see transactions for their own region. We create a row filter function that evaluates group membership at query runtime and restricts rows accordingly.


In [ ]:
%sql
USE CATALOG trainer_demo;
USE SCHEMA demo_04;

-- Row filter function: finance_team and retail_analysts see all regions;
-- region-specific manager groups see only their region.
CREATE OR REPLACE FUNCTION filter_transaction_region(store_region STRING)
  RETURNS BOOLEAN
  COMMENT 'Restricts transaction rows to the calling principal\'s store region'
  RETURN
    is_account_group_member('finance_team')    OR
    is_account_group_member('retail_analysts') OR
    (is_account_group_member('north_store_managers') AND store_region = 'North') OR
    (is_account_group_member('south_store_managers') AND store_region = 'South') OR
    (is_account_group_member('east_store_managers')  AND store_region = 'East')  OR
    (is_account_group_member('west_store_managers')  AND store_region = 'West');

-- Attach the row filter to the transactions table
ALTER TABLE trainer_demo.demo_04.transactions
  SET ROW FILTER trainer_demo.demo_04.filter_transaction_region ON (store_region);


In [ ]:
%sql
-- Query transactions with the row filter active.
-- As metastore admin you see all regions; store managers would see only their region.
SELECT
  t.transaction_id,
  c.first_name || ' ' || c.last_name AS customer_name,
  p.product_name,
  s.store_name,
  t.store_region,
  t.transaction_date,
  t.amount
FROM trainer_demo.demo_04.transactions t
JOIN trainer_demo.demo_04.customers c ON t.customer_id = c.customer_id
JOIN trainer_demo.demo_04.products  p ON t.product_id  = p.product_id
JOIN trainer_demo.demo_04.stores    s ON t.store_id    = s.store_id
ORDER BY t.transaction_date DESC
LIMIT 10;


In [ ]:
%sql

ALTER TABLE trainer_demo.demo_04.transactions
  DROP ROW FILTER;

### 🎯 DEMO: Dynamic View as an Alternative

Instead of attaching functions directly to the table, we can create a **dynamic view** that encapsulates the same masking and filtering logic. Users are granted access to the view — the base table stays hidden.

This approach is useful when:
- You need to join multiple tables with consistent security rules
- Organizational policy requires strict separation between consumers and source tables


In [ ]:
%sql
USE CATALOG trainer_demo;
USE SCHEMA demo_04;

-- Dynamic view: masks PII and restricts rows based on group membership
CREATE OR REPLACE VIEW vw_customers_secure AS
SELECT
  customer_id,
  first_name,
  last_name,
  CASE
    WHEN is_account_group_member('finance_team') THEN email
    ELSE CONCAT(LEFT(email, 2), '***@', SPLIT(email, '@')[1])
  END AS email,
  CASE
    WHEN is_account_group_member('finance_team') THEN phone
    ELSE 'XXX-XXX-' || RIGHT(phone, 4)
  END AS phone,
  region,
  customer_segment
FROM trainer_demo.demo_04.customers
WHERE
  -- finance_team and retail_analysts see all customers;
  -- store managers see only customers in their region
  CASE
    WHEN is_account_group_member('finance_team')    THEN TRUE
    WHEN is_account_group_member('retail_analysts') THEN TRUE
    WHEN is_account_group_member('north_store_managers') AND region = 'North' THEN TRUE
    WHEN is_account_group_member('south_store_managers') AND region = 'South' THEN TRUE
    WHEN is_account_group_member('east_store_managers')  AND region = 'East'  THEN TRUE
    WHEN is_account_group_member('west_store_managers')  AND region = 'West'  THEN TRUE
    ELSE FALSE
  END;

-- Query the secure view
SELECT * FROM vw_customers_secure LIMIT 10;


## Access Azure Key Vault secrets

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/secure-unity-catalog-objects.access-azure-key-vault.png)

### 🎯 DEMO: Access Azure Key Vault Secrets

RetailNow's data platform connects to several external systems: a loyalty PostgreSQL database and an ERP system (Azure SQL Database). Credentials must never appear in notebook code or job configurations.

**The pattern: Azure Key Vault-backed secret scopes**
1. Secrets live in **Azure Key Vault** (managed by the security team)
2. A **secret scope** in Databricks references the Key Vault
3. Code retrieves values at runtime using `dbutils.secrets.get()` — the output is always **[REDACTED]**

**Common retail use cases:**
- JDBC connection strings for loyalty / ERP databases
- API keys for payment processors or third-party data feeds
- Storage account keys for partner file drops


In [ ]:
# List all secret scopes registered in this workspace
# In a real setup you might have scopes like 'retail-db-secrets', 'payment-api-keys'
scopes = dbutils.secrets.listScopes()
for s in scopes:
    print(f"Scope: {s.name}")

# ------------------------------------------------------------
# Retrieve a secret – the value is ALWAYS redacted in output
# ------------------------------------------------------------
# db_password = dbutils.secrets.get(scope="retail-db-secrets", key="loyalty-db-password")
# print(db_password)   # prints: [REDACTED]

# Use in a JDBC connection to the loyalty database
# loyalty_df = (
#     spark.read.format("jdbc")
#     .option("url", "jdbc:postgresql://loyalty-db.retailnow.com:5432/loyalty")
#     .option("dbtable", "loyalty_points")
#     .option("user",     dbutils.secrets.get("retail-db-secrets", "loyalty-db-user"))
#     .option("password", dbutils.secrets.get("retail-db-secrets", "loyalty-db-password"))
#     .load()
# )

# List keys in a scope (returns metadata – NOT secret values)
# keys = dbutils.secrets.list("retail-db-secrets")
# for k in keys: print(k.key)

print("Secrets are always [REDACTED] in notebook output – safe to share notebooks.")


In [ ]:
%sql
-- In Databricks SQL use the secret() function – same redaction behaviour.
-- Example: reference credentials inside a CREATE TABLE / OPTIONS block.

-- CREATE TABLE IF NOT EXISTS trainer_demo.demo_04.erp_orders
-- USING jdbc
-- OPTIONS (
--   url      'jdbc:sqlserver://erp.retailnow.com:1433',
--   dbtable  'orders',
--   user     secret('retail-db-secrets', 'erp-db-user'),
--   password secret('retail-db-secrets', 'erp-db-password')
-- );

SELECT 'Use secret(scope, key) in SQL – output is [REDACTED]' AS note;


## Authenticate data access with service principals

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/secure-unity-catalog-objects.authenticate-data-access-service-principals.png)

### 🎯 DEMO: Authenticate Data Access with Service Principals

RetailNow's raw POS export files land in an ADLS Gen2 account that is not yet registered in Unity Catalog. The nightly ETL job needs to read these files using a **service principal** configured with OAuth2.

**Key security practice**: The client secret is **always** retrieved from a Databricks secret scope — never hardcoded.

> **Trainer Note**: This cell is conceptual. Running it requires a real service principal, storage account, and secret scope. The code pattern demonstrates best practices for production use.


In [ ]:
# ── Service Principal Authentication for ADLS Gen2 ───────────────────────────
# Step 1: Retrieve the client secret from the secret scope (never hard-code it)
# sp_client_secret = dbutils.secrets.get(scope="retail-sp-secrets", key="sp-client-secret")

storage_account  = "retailnowdatalake"
sp_application_id = "<application-client-id>"   # replace with actual App (client) ID
sp_tenant_id      = "<directory-tenant-id>"      # replace with actual Directory (tenant) ID

# Step 2: Configure Spark for OAuth2 client credentials flow
# spark.conf.set(
#     f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
# spark.conf.set(
#     f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
#     "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
# spark.conf.set(
#     f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net",
#     sp_application_id)
# spark.conf.set(
#     f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net",
#     sp_client_secret)
# spark.conf.set(
#     f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
#     f"https://login.microsoftonline.com/{sp_tenant_id}/oauth2/token")

# Step 3: Read raw POS exports using the ABFS driver
# pos_df = spark.read.csv(
#     f"abfss://raw@{storage_account}.dfs.core.windows.net/pos-exports/",
#     header=True, inferSchema=True
# )
# pos_df.show(5)

print("Service principal config pattern: credentials come from secret scope, never hardcoded.")
print("Note: Microsoft recommends managed identities for Unity Catalog storage credentials.")


## Authenticate resource access with managed identities

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/secure-unity-catalog-objects.authenticate-resource-access-managed-identities.png)

### 🎯 DEMO: Authenticate Resource Access with Managed Identities

For production, **managed identities** are the preferred authentication method for Unity Catalog storage credentials. Key advantages over service principals:

- **No secrets to manage** – Azure handles credential rotation automatically
- **Network-restricted storage** – managed identities can access ADLS accounts behind firewalls; service principals cannot
- **Simpler governance** – fewer credentials to audit and rotate

**RetailNow scenario**: An Azure Databricks Access Connector is deployed with a system-assigned managed identity. The identity is granted **Storage Blob Data Contributor** on the retail data lake. A Unity Catalog storage credential references the connector, and an external location exposes a specific container path.

> **Trainer Note**: The SQL commands below require metastore admin privileges. Demonstrate conceptually or in a pre-configured admin environment.


In [ ]:
%sql
-- ── Step 1: Create a storage credential (metastore admin required) ─────────
-- CREATE STORAGE CREDENTIAL retailnow_adls_credential
--   WITH AZURE MANAGED IDENTITY
--   (DIRECTORY '/subscriptions/<sub-id>/resourceGroups/retailnow-rg/providers/Microsoft.Databricks/accessConnectors/retailnow-connector');

-- ── Step 2: Create an external location pointing to ADLS Gen2 ─────────────
-- CREATE EXTERNAL LOCATION IF NOT EXISTS retailnow_raw_data
--   URL 'abfss://raw@retailnowdatalake.dfs.core.windows.net/'
--   WITH (CREDENTIAL retailnow_adls_credential)
--   COMMENT 'Raw POS and e-commerce data exports for RetailNow';

-- ── Step 3: Grant data engineers access ───────────────────────────────────
-- GRANT READ FILES  ON EXTERNAL LOCATION retailnow_raw_data TO `data_engineers`;
-- GRANT WRITE FILES ON EXTERNAL LOCATION retailnow_raw_data TO `data_engineers`;

-- List existing storage credentials and external locations in the metastore
SHOW STORAGE CREDENTIALS;


In [ ]:
%sql
-- List all external locations registered in the metastore
SHOW EXTERNAL LOCATIONS;


---
## Demo Summary

In this demo, we secured the **RetailNow** omnichannel retail platform using Unity Catalog:

| Topic | Demonstrated |
|---|---|
| Query lifecycle | How Unity Catalog intercepts, validates, logs, and scopes every query |
| Access control strategies | GRANT / REVOKE at catalog, schema, and table level; inheritance model |
| Fine-grained access control | Row & Column Security vs Dynamic Views – trade-offs and when to use each |
| Column masking | `mask_phone` and `mask_email` functions attached to `customers` |
| Row filtering | `filter_transaction_region` restricts transaction rows per store region |
| Azure Key Vault secrets | `dbutils.secrets.get()` and `secret()` SQL function – always [REDACTED] |
| Service principal auth | OAuth2 Spark configuration – secret from scope, never hardcoded |
| Managed identity auth | Storage credential + external location via Access Connector |

**Key security principles:**
1. **Never hardcode credentials** – always retrieve from secret scopes at runtime
2. **Least privilege** – grant only the minimum required, at the lowest possible level
3. **Mask, don't duplicate** – column masking and row filtering over table copies
4. **Prefer managed identities** – for Unity Catalog storage credentials in production
5. **Trust the audit log** – `system.access.audit` captures every access event
